In [3]:
import pandas as pd
import numpy as np

# Sahi path do
bitcoin = pd.read_csv('C:/Users/shriv/OneDrive/Desktop/crypto/bitcoin_clean.csv')
bitcoin['date'] = pd.to_datetime(bitcoin['date'])
bitcoin = bitcoin.set_index('date')

print("✅ Data Loaded!")
print("Shape:", bitcoin.shape)

✅ Data Loaded!
Shape: (3248, 8)


## Target Variable

In [4]:
bitcoin['daily_return'] = bitcoin['close'].pct_change()
bitcoin['volatility_7d'] = bitcoin['daily_return'].rolling(7).std()

print("✅ Target Ready!")
print(bitcoin[['close','daily_return','volatility_7d']].head())

✅ Target Ready!
                 close  daily_return  volatility_7d
date                                               
2013-05-05  115.910004           NaN            NaN
2013-05-06  112.300003     -0.031145            NaN
2013-05-07  111.500000     -0.007124            NaN
2013-05-08  113.566002      0.018529            NaN
2013-05-09  112.669998     -0.007890            NaN


## Moving Averages

In [5]:
bitcoin['MA_7']  = bitcoin['close'].rolling(7).mean()
bitcoin['MA_14'] = bitcoin['close'].rolling(14).mean()
bitcoin['MA_30'] = bitcoin['close'].rolling(30).mean()

print("✅ Moving Averages Done!")

✅ Moving Averages Done!


## Price Feature

In [6]:
bitcoin['price_range']     = bitcoin['high'] - bitcoin['low']
bitcoin['price_range_pct'] = bitcoin['price_range'] / bitcoin['close']
bitcoin['close_open_pct']  = (bitcoin['close'] - bitcoin['open']) / bitcoin['open']

print("✅ Price Features Done!")

✅ Price Features Done!


## Bollinger Bands

In [7]:
bitcoin['BB_middle'] = bitcoin['close'].rolling(20).mean()
bitcoin['BB_std']    = bitcoin['close'].rolling(20).std()
bitcoin['BB_upper']  = bitcoin['BB_middle'] + (2 * bitcoin['BB_std'])
bitcoin['BB_lower']  = bitcoin['BB_middle'] - (2 * bitcoin['BB_std'])
bitcoin['BB_width']  = (bitcoin['BB_upper'] - bitcoin['BB_lower']) / bitcoin['BB_middle']

print("✅ Bollinger Bands Done!")

✅ Bollinger Bands Done!


## ATR

In [8]:
bitcoin['TR'] = np.maximum(
    bitcoin['high'] - bitcoin['low'],
    np.maximum(
        abs(bitcoin['high'] - bitcoin['close'].shift(1)),
        abs(bitcoin['low']  - bitcoin['close'].shift(1))
    )
)
bitcoin['ATR_14'] = bitcoin['TR'].rolling(14).mean()

print("✅ ATR Done!")

✅ ATR Done!


## Valume Feature

In [9]:
bitcoin['volume_MA7']    = bitcoin['volume'].rolling(7).mean()
bitcoin['volume_change'] = bitcoin['volume'].pct_change()
bitcoin['volume_spike']  = bitcoin['volume'] / bitcoin['volume_MA7']
bitcoin['liquidity_ratio'] = bitcoin['volume'] / bitcoin['marketCap']

print("✅ Volume Features Done!")

✅ Volume Features Done!


## RSI

In [10]:
def calculate_rsi(series, period=14):
    delta    = series.diff()
    gain     = delta.where(delta > 0, 0)
    loss     = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    rs       = avg_gain / avg_loss
    rsi      = 100 - (100 / (1 + rs))
    return rsi

bitcoin['RSI_14'] = calculate_rsi(bitcoin['close'])

print("✅ RSI Done!")

✅ RSI Done!


## Leg feature

In [11]:
bitcoin['volatility_lag1'] = bitcoin['volatility_7d'].shift(1)
bitcoin['volatility_lag3'] = bitcoin['volatility_7d'].shift(3)
bitcoin['volatility_lag7'] = bitcoin['volatility_7d'].shift(7)
bitcoin['return_lag1']     = bitcoin['daily_return'].shift(1)
bitcoin['return_lag3']     = bitcoin['daily_return'].shift(3)

print("✅ Lag Features Done!")

✅ Lag Features Done!


## Remove NAN

In [13]:
bitcoin.dropna(inplace=True)

print("✅ Shape after cleanup:", bitcoin.shape)
print("✅ Total Features:", bitcoin.shape[1])

bitcoin.to_csv('bitcoin_features.csv')
print("✅ File Save!")

✅ Shape after cleanup: (3012, 33)
✅ Total Features: 33
✅ File Save!
